# Lab — Logistic Regression from scratch
**Machine Learning SUP'COM workshop**

**Goal:** understand logistic regression by building a small class ourselves, then compare it with scikit-learn.

How this lab works:

- Fill in every `# TODO` and replace each `raise NotImplementedError`.
- Run the cells **in order**, from top to bottom.
- Each task ends with a **Check yourself** cell: compare your output with the expected values. If they do not match, fix your code before moving on: later tasks reuse earlier ones.
- Cells marked `# ---- PROVIDED` are complete: just run and read them.
- When you see *Your answer:* ..., write a short answer in that markdown cell.

```text
Problem → Dataset → Visualization → Math idea → Build a class → Train
       → Visualize what it learned → scikit-learn → Compare → Experiments
```

In [1]:
# ---- PROVIDED
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True)

## Part A. The problem

### Task 1. Load and look at the data

> **What are we trying to predict?**

Each example has two features and belongs to one of two classes, $y \in \{0, 1\}$.

In [ ]:
# ---- PROVIDED
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=100,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=2.5,
    random_state=42)



In [ ]:
# TODO (1): print the shape of X and y, the first 5 rows of X, the first 5 labels,
#           and the number of examples per class (hint: np.bincount)
print
# TODO (2): scatter plot of the data, class 0 in blue and class 1 in red.
#           Label the axes "x_1" and "x_2" and add a legend.

> **Can we separate these two groups with a line?**

*Your answer:* ...

**Check yourself:** `X` is (100, 2), `y` is (100,). 50 examples per class. First labels `[0 0 1 0 0]`; first row of `X` is `[2.7283 -2.2572]`.

## Part B. From a line to a probability

### Task 2. The linear score

> **What does the model need to learn?**

$$
z = w^T x + b = w_1 x_1 + w_2 x_2 + b
$$

| Symbol | Meaning |
|---|---|
| $x$ | the input (one row of `X`) |
| $w$ | the **weights**, one per feature |
| $b$ | the **bias** |
| $z$ | the **score**: a single number |

The points where $w^T x + b = 0$ form the **decision boundary**.

In [ ]:
w = np.array([0.5, -1.0])
b = 0.5
x = np.array([2.0, 1.0])

# TODO (1): score of the single example x (use the @ operator)
z = None
print("score for one example:", z)

# TODO (2): scores of the first 5 rows of X, in ONE line, no loop
z_all = None
print("scores for the first 5 examples:", z_all)

To draw the line $w^Tx + b = 0$, solve it for $x_2$:

$$
x_2 = -\frac{w_1 x_1 + b}{w_2}
$$

In [ ]:
x1_line = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)

# TODO: compute x2_line from the formula above, then plot the data and the line
x2_line = None

**Check yourself:** the score of `x` is `0.5`. The first 5 scores are `[4.1213 4.4795 -1.4778 4.3898 4.3804]`. The hand-picked line does **not** separate the classes: we need the computer to learn $w$ and $b$.

### Task 3. The sigmoid

The score $z$ can be any number. It is **not** a probability. The sigmoid squashes it into $(0, 1)$:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad P(y = 1 \mid x) = \sigma(w^T x + b)
$$

In [ ]:
def sigmoid(z):
    # TODO
    raise NotImplementedError


print(sigmoid(0), sigmoid(5), sigmoid(-5))
print(sigmoid(np.array([-2.0, 0.0, 2.0])))     # must also work on arrays

# TODO: plot sigmoid(z) for z in [-8, 8], with dashed lines at y = 0.5 and z = 0

**Check yourself:** `0.5 0.9933 0.0067`, then `[0.1192 0.5 0.8808]`.

> What probability does the model give to a point **exactly on** the decision boundary?

*Your answer:* ...

### Task 4. The loss

> **How do we know if the prediction is good?**

Binary cross-entropy for one example:

$$
L(y, p) = -\left[\, y \log(p) + (1 - y)\log(1 - p) \,\right]
$$

and the cost over $m$ examples:

$$
J(w, b) = \frac{1}{m} \sum_{i=1}^{m} L_i
$$

In [ ]:
def bce(y_true, p):
    # TODO: binary cross-entropy of one example (or element-wise on arrays)
    raise NotImplementedError


for p in [0.99, 0.9, 0.6, 0.5, 0.1, 0.01]:
    print(f"true y = 1, predicted p = {p:<5} -> loss = {bce(1, p):.3f}")

# TODO: on the same figure, plot -log(p) (true y = 1) and -log(1 - p) (true y = 0)
#       for p in [0.001, 0.999]

**Check yourself:** losses `0.010, 0.105, 0.511, 0.693, 2.303, 4.605`.

> Compare a *confident wrong* prediction ($p = 0.01$ for a true class 1) with a *hesitant wrong* one ($p = 0.4$). Why is this behaviour useful?

*Your answer:* ...

## Part C. Build the model

### Task 5. Gradient descent, on paper

> **How can we improve the parameters?**

With $z = Xw + b$ and $p = \sigma(z)$, the gradients of $J$ are:

$$
\nabla_w J = \frac{1}{m} X^T (p - y), \qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m} (p_i - y_i)
$$

and the update, with learning rate $\eta$:

$$
w \leftarrow w - \eta \, \nabla_w J, \qquad
b \leftarrow b - \eta \, \frac{\partial J}{\partial b}
$$

> 1. What is the shape of `X.T @ (p - y)` for our data? Why does it match the shape of `w`?
> 2. How can you read the vector $p - y$ in words?

*Your answer:* ...

### Task 6. The `LogisticRegressionScratch` class

Real ML libraries organise every model the same way:

```python
model = SomeModel(hyperparameters)   # create: nothing learned yet
model.fit(X, y)                      # learn the parameters
model.predict(X_new)                 # use them
```

| Attribute | Set in | Meaning |
|---|---|---|
| `self.lr`, `self.epochs` | `__init__` | hyperparameters: chosen by **us** |
| `self.w`, `self.b` | `fit` | parameters: learned from **data** |
| `self.loss_history` | `fit` | the cost at each epoch |

Before training, the model does not know $w$ and $b$. After `model.fit(X, y)`, the object **contains** them. That is what `self` is for.

Rules: no helper functions outside the class, keep all the maths visible inside `fit`.

In [ ]:
class LogisticRegressionScratch:

    def __init__(self, lr=0.1, epochs=1000):
        # TODO: store the hyperparameters only (self.lr, self.epochs)
        pass

    def sigmoid(self, z):
        # TODO
        raise NotImplementedError

    def fit(self, X, y):
        m, n = X.shape

        # TODO: initialise self.w (zeros, one per feature), self.b (0.0), self.loss_history (empty list)

        # TODO: for each epoch:
        #   1. z = X w + b, p = sigmoid(z)
        #   2. loss = mean binary cross-entropy; clip p to [1e-12, 1 - 1e-12] before the log;
        #      append it to self.loss_history
        #   3. dw and db (formulas of Task 5)
        #   4. update self.w and self.b
        raise NotImplementedError

        return self

    def predict_proba(self, X):
        # TODO: probability of class 1 for each row of X
        raise NotImplementedError

    def predict(self, X):
        # TODO: 1 if the probability is >= 0.5, else 0 (integers)
        raise NotImplementedError

### Task 7. Train it

In [ ]:
model = LogisticRegressionScratch(lr=0.1, epochs=1000)

print("Has w before fit?", hasattr(model, "w"))

# TODO: train the model on (X, y)

print("Has w after fit? ", hasattr(model, "w"))
print("Learned weights w:", model.w)
print("Learned bias b:   ", model.b)

# TODO: proba and pred on X, then the training accuracy
proba = None
pred = None
accuracy = None
print("First 5 probabilities:", proba[:5])
print("First 5 predictions:  ", pred[:5])
print("Training accuracy:", accuracy)

**Check yourself:** `False` then `True`. `w = [-0.0293 2.7944]`, `b ≈ 0.3412`. First probabilities `[0.0024 0.0004 1. 0.0005 0.0005]`, predictions `[0 0 1 0 0]`, accuracy `1.0`.

### Task 8. Watch the training loss

> The optimization process is trying to make this curve go down.

In [ ]:
# TODO: plot model.loss_history (x: epoch, y: cost), then print the first and the last value

**Check yourself:** start `0.6931`, end `0.0037`.

> Why is the very first loss exactly $\log 2 \approx 0.693$?

*Your answer:* ...

## Part D. Visualize what it learned

The function below asks a model to predict on a fine grid and colours each region by its predicted class. It only calls `model.predict`, so it works for **any** model that has that method.

In [ ]:
# ---- PROVIDED
def plot_decision_boundary(model, X, y, title, ax=None):
    if ax is None:
        ax = plt.gca()

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                           np.linspace(x2_min, x2_max, 300))
    grid = np.c_[xx1.ravel(), xx2.ravel()]

    Z = model.predict(grid).reshape(xx1.shape)

    ax.contourf(xx1, xx2, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", label="class 0", edgecolor="k")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red",  label="class 1", edgecolor="k")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.legend()


plot_decision_boundary(model, X, y, "Our scratch logistic regression")
plt.show()

### Task 9. The probability map

Instead of the predicted class, draw the **probability** of class 1 on the grid.

In [ ]:
x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                       np.linspace(x2_min, x2_max, 300))

# TODO (1): P = probability of class 1 for every grid point, reshaped like xx1
P = None

# TODO (2): plt.contourf(xx1, xx2, P, levels=20, cmap="coolwarm", alpha=0.7) + a colorbar,
#           the boundary with plt.contour(xx1, xx2, P, levels=[0.5], colors="k"), then the data

> Where is the model confident, and where is it unsure? Which equation describes the black line?

*Your answer:* ...

## Part E. Compare with scikit-learn

### Task 10. Same interface, production implementation

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO (1): create sk_model = LogisticRegression() and fit it on (X, y)

# TODO (2): sk_pred and sk_proba on X; print sk_model.coef_[0], sk_model.intercept_[0]
#           and the first 3 rows of sk_proba
sk_pred = None
sk_proba = None

**Check yourself:** sklearn `w = [-0.5332 1.9738]`, `b ≈ 1.5772`. First row of `sk_proba`: `[0.987 0.013]`.

> sklearn's `predict_proba` returns **two** columns, ours returns one. What are they, and which column matches ours?

*Your answer:* ...

In [ ]:
# TODO (1): accuracy of both models, number of points where pred == sk_pred,
#           and the max absolute difference between proba and sk_proba[:, 1]

# TODO (2): both boundaries side by side:
#   fig, axes = plt.subplots(1, 2, figsize=(12, 4.5)), then plot_decision_boundary(..., ax=axes[0]) etc.

**Check yourself:** accuracy `1.0` for both, same predictions on `100` of 100 points, max probability difference `0.018`.

> Both models are perfect on the training set, yet the two lines are different. Give two reasons.

*Your answer:* ...

## Part F. Small experiments

For each experiment, **write your prediction first**, then run the cell.

### Task 11. The learning rate

In [ ]:
# TODO: for lr in [0.001, 0.01, 0.1, 1.0], train a model (1000 epochs) and plot its loss_history
#       on the same figure, with a legend

*Your prediction and observation:* ...

### Task 12. Number of epochs

In [ ]:
# TODO: for epochs in [5, 50, 1000], train a model (lr = 0.1) and plot its boundary
#       in a 1x3 figure; put the training accuracy in each title

*Your prediction and observation:* ...

### Task 13. Harder data

In [ ]:
X_hard, y_hard = make_classification(
    n_samples=100, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=0.5, random_state=0
)

# TODO: train a model on (X_hard, y_hard), print its training accuracy and plot its boundary

**Check yourself:** training accuracy `0.86`.

*Can a line still be perfect? What happens to the final loss?* ...

## Summary

$$
z = w^T x + b \;\longrightarrow\; p = \sigma(z) \;\longrightarrow\; \text{BCE loss} \;\longrightarrow\; \text{gradient descent}
$$

Main idea:

> **Model the probability of class 1.**

```python
model = LogisticRegressionScratch(lr=0.1, epochs=1000)
model.fit(X, y)          # learns self.w and self.b
model.predict_proba(X)   # probabilities
model.predict(X)         # classes
```

Next lab: **SVM**. Same problem, same interface, a different idea of what makes a boundary "good".